In [1]:
# Importar librerías
import pandas
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures

In [2]:
data = pandas.read_csv("./IMDb_Limpio.csv")

In [3]:
data.head()

,Unnamed: 0,imdb_title_id,title,original_title,year,date_published,genre,duration,country,language,...,production_company_1967,production_company_2037,production_company_2150,production_company_2240,production_company_2249,production_company_2260,production_company_2286,production_company_2328,production_company_2334,gross_income
0,506,tt0017136,Metropolis,Metropolis,0,1928-10-01,"Drama, Sci-Fi",153,Germany,German,...,0,0,0,0,0,0,0,0,0,28.142926
1,1048,tt0021749,Luci della città,City Lights,1,1931-04-02,"Comedy, Drama, Romance",87,USA,English,...,0,0,0,0,0,0,0,0,0,20.598246
2,2454,tt0027977,Tempi moderni,Modern Times,2,1937-03-12,"Comedy, Drama, Family",87,USA,English,...,0,0,0,0,0,0,0,0,0,25.038982
3,2795,tt0029453,Il bandito della Casbah,Pépé le Moko,3,1937-10-22,"Crime, Drama, Romance",94,France,"French, Arabic",...,0,0,0,0,0,0,0,0,0,23.913876
4,2827,tt0029583,Biancaneve e i sette nani,Snow White and the Seven Dwarfs,3,1938-11-30,"Animation, Family, Fantasy",83,USA,English,...,0,0,0,0,0,0,0,0,0,38.070927


# Variables que se usan para el split de los datos

In [8]:
# Limpieza de Género. Hace dummies para cada género distinto. 
generos = []
for j in data['genre'].dropna().str.rsplit(', '):
    for i in j:
        if i not in generos:
            generos.append(i)

generos_names = ["{}{}".format(i,"_Genre") for i in generos]

lenguaje = []
for j in data['language'].dropna().str.rsplit(', '):
    for i in j:
        if i not in lenguaje:
            lenguaje.append(i)

lenguaje_names = ["{}{}".format(i,"_Language") for i in lenguaje]

pais = []
for j in data['country'].dropna().str.rsplit(', '):
    for i in j:
        if i not in pais:
            pais.append(i)

pais_names = ["{}{}".format(i,"_Country") for i in pais]

In [9]:
director_list = [s for s in list(data.columns) if "director" in s]
writer_list = [s for s in list(data.columns) if "writer" in s]
prodcomp_list = [s for s in list(data.columns) if "production" in s]

# Separacion en training, test y validación 

In [10]:
# Separa la base de datos, usa la variable "quality" como variable objetivo, y
# todas las demás variables como variables explicativas
X = data[["duration", "avg_vote", "votes", \
          "budget","metascore", "reviews_from_users", "reviews_from_critics"]].to_numpy()
X2 = data[generos_names + lenguaje_names + pais_names \
         + director_list + writer_list + prodcomp_list].to_numpy()
Y = data["gross_income"].to_numpy()

print(X.shape)
print(Y.shape)

(6616, 7)
(6616,)


In [11]:
X = PolynomialFeatures(2, interaction_only=True, include_bias=False).fit_transform(X)
X = np.concatenate((X, X2), axis=1)
print(X.shape)
print(Y.shape)

(6616, 721)
(6616,)


In [12]:
# Separamos en training, validación y testing
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=42)

print(f"{len(x_train)} training samples")
print(f"{len(x_val)} validation samples")
print(f"{len(x_test)} test samples")

4762 training samples
530 validation samples
1324 test samples


# Setup para modelar

In [13]:
# Metricas
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import r2_score

# Modelos
from sklearn.neighbors import KNeighborsRegressor # kNN
from sklearn.tree import DecisionTreeRegressor # Arbol de Decisiones
from sklearn.svm import SVR # Support Vector Regression
from sklearn.linear_model import Lasso # Regresion Regularizada con Lasso
from sklearn.ensemble import RandomForestRegressor # Random Forest
from sklearn.neural_network import MLPRegressor # Multi Layer perceptron

# Para poder hacer búsqueda de línea 
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV

In [14]:
# Set print options for decimal values
np.set_printoptions(formatter={'float': lambda x: "{0:0.2f}".format(x)})

# K-vecinos

In [19]:
# Grid search over kNN regressor
parameters = {'n_neighbors': range(1, 2), 'weights': ['uniform', 'distance'],
              'p': range(1, 2), 'algorithm': ['auto', 'ball_tree']}

gs_model = GridSearchCV(KNeighborsRegressor(), parameters, n_jobs=-1, scoring="r2")
gs_model.fit(x_train, y_train)

print(f"Best score: {gs_model.best_score_:.3f}")
print(f"Best params: {gs_model.best_params_}")

Best score: 0.414
Best params: {'algorithm': 'auto', 'n_neighbors': 1, 'p': 1, 'weights': 'uniform'}


In [20]:
# Predict on validation data, and print score
y_val_hat = gs_model.best_estimator_.predict(x_val)

print("prediction: ", y_val_hat[:10])
print("true label: ", y_val[:10].astype('float'))

print(f"MSE on the training set: {mean_squared_error(gs_model.best_estimator_.predict(x_train), y_train):.3f}")
print(f"MSE on the validation set: {mean_squared_error(gs_model.best_estimator_.predict(x_val), y_val):.3f}")
print(f"MSE on the test set: {mean_squared_error(gs_model.best_estimator_.predict(x_test), y_test):.3f}")

prediction:  [30.37 35.72 34.99 25.06 34.72 27.35 31.58 34.73 31.62 35.76]
true label:  [33.61 33.46 33.83 19.08 33.31 22.22 33.88 36.64 33.92 35.92]
MSE on the training set: 0.000
MSE on the validation set: 15.300
MSE on the test set: 16.979
